<!-- Cell 1 -->
# 07 — Export to ONNX

Convert the trained YOLOv11 checkpoint (`weights/best.pt`) to `weights/best.onnx` for deployment in ONNX Runtime, OpenCV DNN, or other runtimes.

In [ ]:
# Cell 2
# Install dependencies
%pip install -q "ultralytics==8.3.*" "onnx>=1.12.0" onnxruntime
print("nb07 dependencies ready")

In [ ]:
# Cell 3
import re
import shutil
from pathlib import Path

from ultralytics import YOLO

In [ ]:
# Cell 4
# ── Run selector ─────────────────────────────────────────────────────────────
# Set RUN_NAME to a specific folder name, or leave None to auto-select latest.
RUN_NAME = '04_training_batch_1_48am_24_04_2026'   # e.g. '01_training_batch_2_00am_23_04_2026'

# ── Paths (derived — edit only RUN_NAME above) ────────────────────────────────
MODEL_OUTPUTS_DIR = Path('../model_outputs').resolve()

if RUN_NAME is None:
    _runs = sorted([
        d for d in MODEL_OUTPUTS_DIR.iterdir()
        if d.is_dir() and re.match(r'^\d+_training_batch_', d.name)
    ])
    assert _runs, f'No training runs found in {MODEL_OUTPUTS_DIR}'
    RUN_DIR  = _runs[-1]
    RUN_NAME = RUN_DIR.name
else:
    RUN_DIR = MODEL_OUTPUTS_DIR / RUN_NAME

WEIGHTS  = (RUN_DIR / 'weights' / 'best.pt').resolve()
ONNX_OUT = WEIGHTS.with_suffix('.onnx')

IMGSZ    = 640
OPSET    = 12
DYNAMIC  = True    # dynamic batch/HW axes
SIMPLIFY = True    # run onnx-simplifier
HALF     = False   # fp16 export (GPU only)

assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS} — run notebook 04 first'
print(f'Run     : {RUN_NAME}')
print(f'Source  : {WEIGHTS}')
print(f'Target  : {ONNX_OUT}')

In [ ]:
# Cell 5
# ── Export ─────────────────────────────────────────────────────────────────────────
model = YOLO(str(WEIGHTS))
exported = model.export(
    format='onnx',
    imgsz=IMGSZ,
    opset=OPSET,
    dynamic=DYNAMIC,
    simplify=SIMPLIFY,
    half=HALF,
)
exported = Path(exported)

# Ultralytics writes <stem>.onnx next to the .pt — move it to ONNX_OUT if needed
if exported.resolve() != ONNX_OUT:
    shutil.move(str(exported), ONNX_OUT)

print(f'Exported: {ONNX_OUT}  ({ONNX_OUT.stat().st_size/1e6:.1f} MB)')

In [ ]:
# Cell 6
# ── Verify ─────────────────────────────────────────────────────────────────────────
import numpy as np
import onnx
import onnxruntime as ort

onnx_model = onnx.load(str(ONNX_OUT))
onnx.checker.check_model(onnx_model)

sess = ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print(f'Input   : {inp.name}  shape={inp.shape}  dtype={inp.type}')
print(f'Output  : {out.name}  shape={out.shape}  dtype={out.type}')

dummy = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)
pred = sess.run(None, {inp.name: dummy})[0]
print(f'Dummy forward pass OK \u2014 output shape: {pred.shape}')